In [ ]:
! pip install ultralytics opencv-python pyyaml numpy pandas scikit-learn torch matplotlib seaborn paddleocr paddlepaddle

In [ ]:
import cv2, os, random, shutil, zipfile, yaml, csv, math, paddle, threading, time, json, subprocess
import numpy as np
from pathlib import Path
from tqdm import tqdm
from typing import List, Tuple, Optional, Dict
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from uuid import uuid4
from concurrent.futures import ThreadPoolExecutor, as_completed
from ultralytics import YOLO
from paddleocr import PaddleOCR


# configs
BASE_DIR = Path("../dataset/anpr/")
ARCHIVE_FILE = Path("../dataset/anpr/archive.zip")
DATASET_DIR = (BASE_DIR / "cleaned")
YOLO_DIR = (BASE_DIR / "yolo_processed")
OCR_DIR = (BASE_DIR / "ocr_processed")

YAML_PATH = ("data.yaml")
DATA_YAML = (YOLO_DIR / "data.yaml")
MODEL_ARCH = ("../models/yolov8m.pt")

FOLDERS = ["google_images", "video_images", "State-wise_OLX"]
PRO_FOLDERS = ["train", "valid", "test"]
SPLIT_RATIOS = (0.8, 0.1, 0.1)

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
PRO_IMG_EXTENSIONS = {".jpg", ".jpeg", ".png"}
ANNOTATION_EXT = ".xml"
YOLO_ANNOTATION_EXT = ".txt"

RUN_NAME = "anpr_v1"
RUN_NAME2 = "anpr_v2"
EPOCHS_DRY = 1
EPOCHS_HALF = 10
EPOCHS_FULL = 50
IMG_SIZE = 640
SAMPLES_TO_VISUALIZE = 5

In [ ]:
colab_content = "/content/"
folders = [
    "models",
    "configs",
    "en_PP-OCRv3_rec_slim_train"
]

# Create them
for folder in folders:
    full_path = os.path.join(colab_content, folder)
    os.makedirs(full_path, exist_ok=True)
    print(f"[INFO] Created: {full_path}")

In [ ]:
class DatasetPreparer:
    def __init__(self, base_dir: Path, archive_file: Path, dataset_dir: Path, yolo_dir: Path, ocr_dir: Path,
                 folders: List[str], pro_folders: List[str], split_ratios: Tuple[float, float, float],
                 img_extensions=(".jpg", ".jpeg", ".png"), annotation_ext=".xml", yolo_annotation_ext=".txt",
                 yaml_path="data.yaml"):

        self.BASE_DIR = base_dir
        self.ARCHIVE_FILE = archive_file
        self.DATASET_DIR = dataset_dir
        self.YOLO_DIR = yolo_dir
        self.OCR_DIR = ocr_dir
        self.FOLDERS = folders
        self.PRO_FOLDERS = pro_folders
        self.SPLIT_RATIOS = split_ratios
        self.IMG_EXTENSIONS = img_extensions
        self.ANNOTATION_EXT = annotation_ext
        self.YOLO_ANNOTATION_EXT = yolo_annotation_ext
        self.YAML_PATH = yaml_path

    # ---------- STAGE 1 ----------
    def extract_archive(self):
        try:
            self.BASE_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(self.ARCHIVE_FILE, 'r') as zip_ref:
                zip_ref.extractall(self.BASE_DIR)
            print(f"[INFO] Extracted archive: {self.ARCHIVE_FILE} -> {self.BASE_DIR}")
        except Exception as e:
            print(f"[ERROR] Failed to extract archive: {e}")

    def collect_images(self):
        all_images = []
        try:
            for folder in self.FOLDERS:
                folder_path = self.BASE_DIR / folder
                if not folder_path.exists():
                    print(f"[WARN] Folder not found: {folder_path}")
                    continue
                for root, _, files in os.walk(folder_path):
                    for file in files:
                        if Path(file).suffix.lower() in self.IMG_EXTENSIONS:
                            all_images.append(Path(root) / file)
            print(f"[INFO] Total images collected: {len(all_images)}")
        except Exception as e:
            print(f"[ERROR] Failed to collect images: {e}")
        return all_images

    def split_dataset(self, images):
        try:
            random.shuffle(images)
            n_total = len(images)
            n_train = int(self.SPLIT_RATIOS[0] * n_total)
            n_valid = int(self.SPLIT_RATIOS[1] * n_total)

            train_set = images[:n_train]
            valid_set = images[n_train:n_train + n_valid]
            test_set = images[n_train + n_valid:]
            print(f"[INFO] Split: Train={len(train_set)}, Valid={len(valid_set)}, Test={len(test_set)}")
            return train_set, valid_set, test_set
        except Exception as e:
            print(f"[ERROR] Failed to split dataset: {e}")
            return [], [], []

    def copy_files_with_annotations(self, images, target_dir):
        try:
            target_dir.mkdir(parents=True, exist_ok=True)
            for img_path in images:
                try:
                    shutil.copy(img_path, target_dir / img_path.name)
                    annotation_path = img_path.with_suffix(self.ANNOTATION_EXT)
                    if annotation_path.exists():
                        shutil.copy(annotation_path, target_dir / annotation_path.name)
                    else:
                        print(f"[WARN] No annotation for {img_path.name}")
                except Exception as e:
                    print(f"[ERROR] Failed to copy {img_path.name}: {e}")
        except Exception as e:
            print(f"[ERROR] Failed to copy files to {target_dir}: {e}")

    def prepare(self):
        # Step 0: Extract archive
        self.extract_archive()

        # Step 1: collect all imgs
        images = self.collect_images()
        if not images:
            print("[ERROR] No images found. Exiting...")
            return

        # Step 2: split dataset
        train_set, valid_set, test_set = self.split_dataset(images)

        # Step 3: copy files
        print("[INFO] Copying Train set...")
        self.copy_files_with_annotations(train_set, self.DATASET_DIR / "train")
        print("[INFO] Copying Valid set...")
        self.copy_files_with_annotations(valid_set, self.DATASET_DIR / "valid")
        print("[INFO] Copying Test set...")
        self.copy_files_with_annotations(test_set, self.DATASET_DIR / "test")
        print("[INFO] Dataset prepared successfully!")

    def check_annotations(self, split_dir):
        try:
            print(f"\n[INFO] Checking directory: {split_dir}")
            images = [f for f in split_dir.iterdir() if f.suffix.lower() in self.IMG_EXTENSIONS]
            annotations = [f for f in split_dir.iterdir() if f.suffix.lower() == self.ANNOTATION_EXT]

            image_names = {f.stem for f in images}
            annotation_names = {f.stem for f in annotations}

            missing_annotations = image_names - annotation_names
            missing_images = annotation_names - image_names

            print(f"  Total Images: {len(images)}")
            print(f"  Total Annotations: {len(annotations)}")

            if missing_annotations:
                print(f"  [WARN] Missing annotations for {len(missing_annotations)} images:")
                for name in sorted(missing_annotations):
                    print(f"    - {name}")
            else:
                print("  [INFO] All images have corresponding annotations.")

            if missing_images:
                print(f"  [WARN] Found annotations with no matching images:")
                for name in sorted(missing_images):
                    print(f"    - {name}")
        except Exception as e:
            print(f"[ERROR] Failed to check {split_dir}: {e}")

    def verify(self):
        for split in ["train", "valid", "test"]:
            split_dir = self.DATASET_DIR / split
            if not split_dir.exists():
                print(f"[WARN] Split directory not found: {split_dir}")
                continue
            self.check_annotations(split_dir)
        print("\n[INFO] Dataset verification complete.")

    # ---------- STAGE 2 ----------
    def create_yolo_dirs(self):
        for split in self.PRO_FOLDERS:
            (self.YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
            (self.YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

    def convert_xml_to_yolo(self, xml_file, label_path):
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
            img_w = int(root.find("size/width").text)
            img_h = int(root.find("size/height").text)

            yolo_labels = []
            for obj in root.findall("object"):
                bbox = obj.find("bndbox")
                if bbox is None:
                    continue
                xmin = float(bbox.find("xmin").text)
                ymin = float(bbox.find("ymin").text)
                xmax = float(bbox.find("xmax").text)
                ymax = float(bbox.find("ymax").text)

                x_center = ((xmin + xmax) / 2) / img_w
                y_center = ((ymin + ymax) / 2) / img_h
                w = (xmax - xmin) / img_w
                h = (ymax - ymin) / img_h

                yolo_labels.append(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

            if not yolo_labels:
                print(f"[WARN] No objects found in {xml_file}")

            with open(label_path, "w") as f:
                f.write("\n".join(yolo_labels))
        except Exception as e:
            print(f"[ERROR] Failed to convert {xml_file}: {e}")

    def preprocess_yolo(self):
        print("[INFO] Starting pre-processing...")
        self.create_yolo_dirs()
        print("[INFO] Created necessary directories for YOLO format.")

        for split in self.PRO_FOLDERS:
            img_dir = self.DATASET_DIR / split
            for img_file in img_dir.iterdir():
                if img_file.suffix.lower() not in self.IMG_EXTENSIONS:
                    continue

                xml_file = img_file.with_suffix(self.ANNOTATION_EXT)
                target_img = self.YOLO_DIR / "images" / split / img_file.name
                target_lbl = self.YOLO_DIR / "labels" / split / (img_file.stem + self.YOLO_ANNOTATION_EXT)

                shutil.copy(img_file, target_img)
                if xml_file.exists():
                    self.convert_xml_to_yolo(xml_file, target_lbl)
                else:
                    print(f"[WARN] No annotation found for {img_file.name}, creating empty label.")
                    open(target_lbl, "w").close()
        print("[INFO] YOLO pre-processing completed!")

    def validate_yolo(self):
        print("[INFO] Starting YOLO dataset validation...")
        for split in self.PRO_FOLDERS:
            img_dir = self.YOLO_DIR / "images" / split
            lbl_dir = self.YOLO_DIR / "labels" / split
            images = [f for f in img_dir.glob("*.*")]
            labels = [f for f in lbl_dir.glob("*.*")]
            image_names = {f.stem for f in images}
            label_names = {f.stem for f in labels}

            missing_labels = image_names - label_names
            extra_labels = label_names - image_names

            print(f"\n[INFO] Split: {split}")
            print(f"  Images: {len(image_names)}, Labels: {len(label_names)}")
            if missing_labels:
                print(f"  [WARN] Missing labels: {missing_labels}")
                for name in sorted(missing_labels):
                    print(f"    - {name}")
            if extra_labels:
                print(f"  [WARN] Extra labels: {extra_labels}")
                for name in sorted(extra_labels):
                    print(f"    - {name}")
        print("[INFO] YOLO validation complete.")

    def generate_data_yaml(self):
        data_config = {
            "train": str((self.YOLO_DIR / "images" / "train").resolve()),
            "val": str((self.YOLO_DIR / "images" / "valid").resolve()),
            "test": str((self.YOLO_DIR / "images" / "test").resolve()),
            "nc": 1,
            "names": ["license_plate"]
        }
        yaml_path = self.YOLO_DIR / self.YAML_PATH
        with open(yaml_path, "w") as f:
            yaml.dump(data_config, f)
        print(f"[INFO] data.yaml created at: {yaml_path}")
        return yaml_path

    # ---------- STAGE 3 ----------
    def parse_annotation(self, xml_file: Path):
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
            image_file = root.find("filename").text

            annotations = []
            for obj in root.findall("object"):
                text = obj.find("name").text.strip()
                bbox = obj.find("bndbox")
                xmin = int(float(bbox.find("xmin").text))
                ymin = int(float(bbox.find("ymin").text))
                xmax = int(float(bbox.find("xmax").text))
                ymax = int(float(bbox.find("ymax").text))

                points = [[xmin, ymin], [xmax, ymin], [xmax, ymax], [xmin, ymax]]
                annotations.append({"transcription": text, "points": points})

            return image_file, annotations
        except Exception as e:
            print(f"[ERROR] Failed to parse {xml_file}: {e}")
            return None, None

    def create_split_file(self, split, output_txt, images_out_dir):
        split_dir = self.DATASET_DIR / split
        images_out_dir.mkdir(parents=True, exist_ok=True)

        with open(output_txt, "w", encoding="utf-8") as f:
            for xml_file in split_dir.glob("*.xml"):
                image_name, annotations = self.parse_annotation(xml_file)
                if not image_name or not annotations:
                    continue

                image_path = split_dir / image_name
                if not image_path.exists():
                    print(f"[WARN] Missing image file: {image_path}")
                    continue

                shutil.copy(image_path, images_out_dir / image_name)
                f.write(f"{str(images_out_dir / image_name)}\t{json.dumps(annotations, ensure_ascii=False)}\n")
        print(f"[INFO] Created PaddleOCR label: {output_txt}")

    def create_paddleocr_data(self):
        labels_dir = self.OCR_DIR / "labels"
        images_dir = self.OCR_DIR / "images"
        labels_dir.mkdir(parents=True, exist_ok=True)

        for split in self.PRO_FOLDERS:
            self.create_split_file(split, labels_dir / f"{split}.txt", images_dir / split)

    def verify_paddleocr_data(self):
        for split in self.PRO_FOLDERS:
            images_dir = self.OCR_DIR / "images" / split
            label_file = self.OCR_DIR / "labels" / f"{split}.txt"
            if not label_file.exists():
                print(f"[ERROR] Missing label file: {label_file}")
                continue

            image_paths = list(images_dir.glob("*.*"))
            label_lines = label_file.read_text(encoding="utf-8").splitlines()
            label_img_names = {Path(line.split("\t")[0]).name for line in label_lines if "\t" in line}
            image_names = {p.name for p in image_paths}

            print(f"\n[INFO] Verifying {split}: {len(image_names)} images, {len(label_img_names)} labels")
            print(f"  Unused images: {image_names - label_img_names}")
            print(f"  Missing images: {label_img_names - image_names}")

In [ ]:
prep = DatasetPreparer(
    base_dir=Path("/content/data/"),
    archive_file=Path("/content/archive.zip"),
    dataset_dir=Path("/content/data/cleaned"),
    yolo_dir=Path("/content/data/yolo_processed"),
    ocr_dir=Path("/content/data/ocr_processed"),
    folders=["google_images", "video_images", "State-wise_OLX"],
    pro_folders=["train", "valid", "test"],
    split_ratios=(0.8, 0.1, 0.1)
)

prep.prepare()               # Stage 1
prep.verify()                # Verify Stage 1
prep.preprocess_yolo()       # Stage 2
prep.validate_yolo()         # Validate YOLO
prep.generate_data_yaml()    # Create data.yaml
prep.create_paddleocr_data() # Stage 3
prep.verify_paddleocr_data() # Verify OCR

In [ ]:
# STEP 1: SANITY CHECK (VISUALIZE ANNOTATIONS)
def visualize_samples(data_yaml, pro_dir, yolo_annotation_ext, samples):

    dataset_path = pro_dir / "images" / "train"
    label_path = pro_dir / "labels" / "train"
    all_images = list(dataset_path.glob("*.jpg")) + list(dataset_path.glob("*.jpeg")) + list(dataset_path.glob("*.png"))

    random.shuffle(all_images)

    print(f"[INFO] Visualizing {samples} random samples with bounding boxes...")
    for img_file in all_images[:samples]:
        label_file = label_path / (img_file.stem + yolo_annotation_ext)
        img = cv2.imread(str(img_file))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if label_file.exists():
            h, w, _ = img.shape
            with open(label_file, "r") as f:
                lines = f.readlines()
                for line in lines:
                    cls, x, y, bw, bh = map(float, line.strip().split())
                    x, y, bw, bh = x * w, y * h, bw * w, bh * h
                    x1, y1 = int(x - bw/2), int(y - bh/2)
                    x2, y2 = int(x + bw/2), int(y + bh/2)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    cv2.putText(img, "license_plate", (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

        plt.imshow(img)
        plt.axis("off")
        plt.show()

# STEP 2: DRY RUN TRAINING
def dry_run_training(data_yaml, model_arch, epochs, imgsz, run_name):
    print(f"[INFO] Starting Dry Run (epochs={epochs})...")
    model = YOLO(model_arch)
    model.train(
        data=data_yaml,
        epochs=epochs,
        imgsz=imgsz,
        project="runs/detect",
        name=run_name + "_dry",
        verbose=True,
        plots=True,
    )
    print("[INFO] Dry Run Completed. Check runs/detect folder.")

# STEP 3: FULL TRAINING
def full_training(data_yaml, model_arch, epochs, imgsz, run_name):
    print(f"[INFO] Starting Full Training (epochs={epochs})...")
    model = YOLO(model_arch)
    model.train(
        data=data_yaml,
        epochs=epochs,
        imgsz=imgsz,
        project="runs/detect",
        name=run_name,
        verbose=True,
        plots=True,
    )
    print("[INFO] Full Training Completed. Check runs/detect folder.")

In [ ]:
# STEP 1: Visualize Sample Images
visualize_samples(DATA_YAML, YOLO_DIR, YOLO_ANNOTATION_EXT, SAMPLES_TO_VISUALIZE)

In [ ]:
# STEP 2: Dry Run
dry_run_training(DATA_YAML, MODEL_ARCH, EPOCHS_DRY, IMG_SIZE, RUN_NAME)

In [ ]:
# STEP 3: Half Training
full_training(DATA_YAML, MODEL_ARCH, EPOCHS_HALF, IMG_SIZE, RUN_NAME)

In [ ]:
# STEP 4: Full Training
full_training(DATA_YAML, MODEL_ARCH, EPOCHS_FULL, IMG_SIZE, RUN_NAME2)

In [ ]:
class DatasetPreparerFlatten:
    def __init__(self, base_dir: str):
        self.base_dir = base_dir
        self.labels_dir = os.path.join(base_dir, "labels")
        self.images_dir = os.path.join(base_dir, "images")

    def flatten_ppocr_labels(self, split: str):
        input_txt_path = os.path.join(self.labels_dir, f"{split}.txt")
        output_txt_path = os.path.join(self.labels_dir, f"{split}_flat.txt")

        with open(input_txt_path, 'r', encoding='utf-8') as infile, open(output_txt_path, 'w', encoding='utf-8') as outfile:
            for line in infile:
                parts = line.strip().split('\t')
                if len(parts) != 2:
                    print(f"Skipping malformed line: {line}")
                    continue

                image_path, json_label = parts
                try:
                    data = json.loads(json_label)
                    if isinstance(data, list) and "transcription" in data[0]:
                        text = data[0]["transcription"]
                        outfile.write(f"{image_path}\t{text}\n")
                    else:
                        print(f"Skipping label without 'transcription': {line}")
                except json.JSONDecodeError:
                    print(f"JSON decode error for line: {line}")

        print(f"[✔] Flattened label file saved to: {output_txt_path}")

    def flatten_all(self):
        for split in ["train", "valid", "test"]:
            self.flatten_ppocr_labels(split)

    def move_samples_from_test_to_valid(self, n: int = 150):
        test_file = os.path.join(self.labels_dir, "test_flat.txt")
        valid_file = os.path.join(self.labels_dir, "valid_flat.txt")
        test_img_dir = os.path.join(self.images_dir, "test")
        valid_img_dir = os.path.join(self.images_dir, "valid")

        with open(test_file, "r", encoding="utf-8") as f:
            test_lines = f.readlines()

        moved_lines = random.sample(test_lines, n)
        updated_moved_lines = []

        for line in moved_lines:
            img_path, label = line.strip().split("\t")
            img_name = os.path.basename(img_path)

            src_img_path = os.path.join(test_img_dir, img_name)
            dst_img_path = os.path.join(valid_img_dir, img_name)

            if os.path.exists(src_img_path):
                shutil.move(src_img_path, dst_img_path)
            else:
                print(f"⚠️ Image not found: {src_img_path}")
                continue

            updated_line = f"{dst_img_path}\t{label}\n"
            updated_moved_lines.append(updated_line)

        remaining_test_lines = [line for line in test_lines if line not in moved_lines]

        with open(valid_file, "a", encoding="utf-8") as f:
            f.writelines(updated_moved_lines)

        with open(test_file, "w", encoding="utf-8") as f:
            f.writelines(remaining_test_lines)

        print(f"✅ Moved {len(updated_moved_lines)} image-label pairs to validation.")

    def count_lines(self, filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            return len(f.readlines())

    def count_images(self, folder):
        valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
        return len([f for f in os.listdir(folder) if f.lower().endswith(valid_exts)])

    def report_dataset_stats(self):
        print("📝 Label Line Counts:")
        for split in ["train", "valid", "test"]:
            flat_label_path = os.path.join(self.labels_dir, f"{split}_flat.txt")
            print(f"🔹 {split}_flat.txt: {self.count_lines(flat_label_path)} lines")

        print("\n🖼️ Image Counts:")
        for split in ["train", "valid", "test"]:
            img_folder = os.path.join(self.images_dir, split)
            print(f"📂 {split}: {self.count_images(img_folder)} images")

In [ ]:
base_dir_flatten = "/content/data/ocr_processed"
preparer = DatasetPreparerFlatten(base_dir_flatten)

preparer.flatten_all()
preparer.move_samples_from_test_to_valid(n=150)
preparer.report_dataset_stats()

In [ ]:
shutil.rmtree('/content/sample_data')

In [ ]:
def unzip_data(zip_path: str, extract_to: str):
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"[INFO] Unzipped {zip_path} to {extract_to}")
    except Exception as e:
        print(f"[ERROR] Failed to unzip {zip_path}: {e}")

ZIP_PATH = "/content/ocr_processed_final.zip"
EXTRACT_TO = "/content/data/"

unzip_data(ZIP_PATH, EXTRACT_TO)

In [ ]:
def add_suffix_to_paths(label_file_path):
    updated_lines = []
    
    with open(label_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 2:
                print(f"Skipping malformed line: {line}")
                continue

            img_path, label = parts
            base, ext = os.path.splitext(img_path)
            new_img_path = f"{base}_0{ext}"
            updated_lines.append(f"{new_img_path}\t{label}\n")

    with open(label_file_path, 'w', encoding='utf-8') as f:
        f.writelines(updated_lines)
    print(f"[✔] Updated file: {label_file_path}")

label_dir = "/content/ocr_processed/labels"
for file_name in ["train_final.txt", "valid_final.txt", "test_final.txt"]:
    full_path = os.path.join(label_dir, file_name)
    add_suffix_to_paths(full_path)

In [ ]:
class ANPRPipeline:
    def _load_models(self):
        self.yolo_model = YOLO(str(self.model_path))

    def _reset_stats(self):
        self.stats = {
            'total_processed': 0,
            'successful_ocr': 0,
            'failed_ocr': 0,
            'no_detections': 0
        }

    def __init__(self, model_path: str, output_dir: str, csv_path: str):
        self.model_path = Path(model_path)
        self.output_dir = Path(output_dir)
        self.csv_path = Path(csv_path)

        self._local = threading.local()  # For OCR instance
        self._load_models()
        self._reset_stats()

    def _get_ocr_instance(self):
        if not hasattr(self._local, 'ocr'):
            self._local.ocr = PaddleOCR(
                # use_doc_orientation_classify=False,
                # use_doc_unwarping=False,
                use_textline_orientation=False,
                lang='en',
                text_det_limit_side_len=1280,
                text_det_limit_type='max',
                text_recognition_batch_size=6,
            )
        return self._local.ocr

    def _validate_crop(self, cropped_img: np.ndarray, min_area: int = 100) -> bool:
        if cropped_img is None or cropped_img.size == 0:
            return False

        h, w = cropped_img.shape[:2]
        if h < 10 or w < 20 or (h * w) < min_area:
            return False

        # Check if image is not completely black or white
        mean_val = np.mean(cropped_img)
        if mean_val < 5 or mean_val > 250:
            return False

        return True

    def show_images(self, images: List[np.ndarray], titles: List[str]):
        plt.figure(figsize=(15, 5))
        for i, (img, title) in enumerate(zip(images, titles)):
            plt.subplot(1, len(images), i + 1)
            if len(img.shape) == 2:  # grayscale
                plt.imshow(img, cmap='gray')
            else:
                plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            plt.title(title)
            plt.axis('off')
        plt.show()

    def _convert_to_gray(self, img):
        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    def _apply_clahe(self, img):
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        return clahe.apply(img)

    def _apply_histeq(self, img):
        return cv2.equalizeHist(img)

    def _denoise(self, img):
        return cv2.bilateralFilter(img, 9, 75, 75)

    def _apply_morph(self, img):
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 2))
        return cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)

    def preprocess_plate_variants(self, cropped_img: np.ndarray) -> Optional[Dict[str, np.ndarray]]:
        try:
            if not self._validate_crop(cropped_img):
                return None

            try:
                variants = {"raw": cropped_img}

                gray = self._convert_to_gray(cropped_img)
                variants["gray"] = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

                clahe_img = self._apply_clahe(gray)
                variants["gray + clahe"] = cv2.cvtColor(clahe_img, cv2.COLOR_GRAY2BGR)

                hist_eq = self._apply_histeq(gray)
                variants["gray + histeq"] = cv2.cvtColor(hist_eq, cv2.COLOR_GRAY2BGR)

                denoised = self._denoise(clahe_img)
                variants["clahe + denoised"] = cv2.cvtColor(denoised, cv2.COLOR_GRAY2BGR)

                morph_img = self._apply_morph(clahe_img)
                variants["morphological"] = cv2.cvtColor(morph_img, cv2.COLOR_GRAY2BGR)
            except Exception as e:
                print(f"[ERROR] Preprocessing failed: {e}")

            self.show_images(
                [
                    cropped_img,
                    variants["gray"],
                    variants["gray + clahe"],
                    variants["gray + histeq"],
                    variants["clahe + denoised"],
                    variants["morphological"],
                ],
                [
                    "Cropped Img",
                    "Gray",
                    "Gray + CLAHE",
                    "Gray + Hist. Eq. Filter",
                    "CLACHE + Denoised",
                    "Morphological Gradient",
                ]
            )

            return variants
        except Exception as e:
            print(f"[DEBUG] Preprocessing variants failed: {e}")
            return None

    def _parse_dict_style(self, result):
        if isinstance(result, list) and len(result) > 0 and isinstance(result[0], dict):
            if 'rec_texts' in result[0] and 'rec_scores' in result[0]:
                texts = result[0]['rec_texts']
                scores = result[0]['rec_scores']
                if texts and scores:
                    return texts[0], scores[0]
        raise ValueError("Invalid dict-style format")

    def _parse_list_style(self, result):
        if isinstance(result, list) and len(result) > 0 and isinstance(result[0], list):
            for item in result[0]:
                if isinstance(item, (list, tuple)) and len(item) >= 2:
                    text_info = item[1] if len(item) > 1 else item[0]
                    if isinstance(text_info, (list, tuple)) and len(text_info) >= 2:
                        return text_info[0], text_info[1]
        raise ValueError("Invalid list-style format")

    def _parse_fallback_style(self, result) -> Tuple[str, float]:
        if isinstance(result, list) and len(result) > 0:
            item = result[0]
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                return item[0], item[1]
        raise ValueError("Invalid fallback format")

    def parse_ocr_result(self, result, variant_name: str):
        for parser in [self._parse_dict_style, self._parse_list_style, self._parse_fallback_style]:
            try:
                text, conf = parser(result)
                if text and isinstance(conf, (float, int)) and conf > 0:
                    return str(text), float(conf)
            except Exception:
                continue
        print(f"[DEBUG] Could not parse OCR result for '{variant_name}'")
        return "", 0.0

    def _select_best_result(self, variant_results, current_best_text, current_best_conf, current_best_variant):
        best_text, best_score, best_variant = current_best_text, current_best_conf, current_best_variant
        for text, score, variants in variant_results:
            if score > best_score:
                best_text, best_score = text, score
        return best_text, best_score, best_variant

    def analyze_and_combine_results(self, variant_results, current_best_text, current_best_conf, current_best_variant):
        if not variant_results:
            return current_best_text, current_best_conf, current_best_variant

        return self._select_best_result(variant_results, current_best_text, current_best_conf, current_best_variant)

    def clean_plate_text(self, text: str) -> str:
        if not text:
            return ""

        # Remove common OCR artifacts
        cleaned = text.strip()

        # Remove spaces and special characters (license plates typically don't have them)
        cleaned = ''.join(c for c in cleaned if c.isalnum())

        return cleaned.upper()

    def _run_ocr_on_variants(self, variants):
        results = []
        best_text, best_conf, best_variant = "", 0.0, ""

        ocr = self._get_ocr_instance()
        for name, img in variants.items():

            try:
                result = ocr.predict(img)
                if not result or not result[0]:
                    print(f"[DEBUG] OCR returned no results for '{name}'.")
                    continue

                text, conf = self.parse_ocr_result(result, name)
                if text and conf > 0:
                    cleaned_text = self.clean_plate_text(text)
                    print(f"[{name.upper()}] -> Text: '{text}' -> Cleaned: '{cleaned_text}', Conf: {conf:.4f}")

                    results.append((cleaned_text, conf, name))
                    if conf > best_conf and len(cleaned_text) > 0:
                            best_text, best_conf, best_variant = cleaned_text, conf, name
            except Exception as e:
                print(f"[ERROR] OCR failed for variant '{name}': {e}")
        return results, best_text, best_conf, best_variant

    def safe_ocr_predict_variants(self, cropped_img):
        if cropped_img is None:
            return "", 0.0, ""

        variants = self.preprocess_plate_variants(cropped_img)
        if not variants:
            return "", 0.0, ""

        variant_results, text, conf, name = self._run_ocr_on_variants(variants)
        best_text, best_conf, best_variant = text or "", conf or 0.0, name or ""

        if best_text and len(best_text) < 6:  # Most license plates have 6+ characters
            print(f"[INFO] '{best_text}' using '{best_variant}' seems short. So now, analyzing all results...")
            best_text, best_conf, best_variant = self.analyze_and_combine_results(variant_results, best_text, best_conf, best_variant)

        return best_text, best_conf, best_variant, variants

    def _pad_and_crop(self, img: np.ndarray, box: np.ndarray, img_path: str):
        try:
            # Handle both 4-element (x1,y1,x2,y2) and 5-element (x1,y1,x2,y2,conf) boxes
            if len(box) >= 4:
                x1, y1, x2, y2 = map(int, box[:4])
            else:
                print(f"[ERROR] Invalid box format: {box}")
                return Path(img_path).name, ""

            # Validate bounding box
            h, w = img.shape[:2]

            # Add padding to avoid cutting off characters (especially last one)
            padding_x = int((x2 - x1) * 0.05)  # 5% horizontal padding
            padding_y = int((y2 - y1) * 0.1)   # 10% vertical padding

            x1 = max(0, x1 - padding_x)
            y1 = max(0, y1 - padding_y)
            x2 = min(w, x2 + padding_x)
            y2 = min(h, y2 + padding_y)

            if x2 <= x1 or y2 <= y1:
                return Path(img_path).name, ""

            # Return extracted crop
            return img[y1:y2, x1:x2]
        except Exception as e:
            print(f"[ERROR] Detection processing failed for {img_path}: {e}")
            return Path(img_path).name, ""

    def _save_debug_images(self, img_path: str, det_idx: int, tag: str, result, sub_dir_name: str="ocr_debug_images"):
        debug_subdir = self.output_dir / (sub_dir_name)
        debug_subdir.mkdir(parents=True, exist_ok=True)

        debug_tag_subsubdir = debug_subdir / str(tag)
        debug_tag_subsubdir.mkdir(parents=True, exist_ok=True)

        debug_path = debug_tag_subsubdir / (f"{Path(img_path).stem}_{det_idx}.jpg")
        print(f"[INFO] Debug image saved to: '{debug_path}")
        return cv2.imwrite(str(debug_path), result)

    def process_single_detection(self, img: np.ndarray, box: np.ndarray, img_path: str, detection_idx: int, conf_threshold_rec) -> Tuple[str, str]:
        try:
            print(f"[INFO] Running OCR on {Path(img_path).name}...")
            cropped = self._pad_and_crop(img, box, img_path)
            # Perform variant-based OCR
            text, conf, variant_name, variants = self.safe_ocr_predict_variants(cropped)

            # Debug: Save the cropped region to visually inspect
            debug_crop_path = self._save_debug_images(img_path, detection_idx, "crop", cropped)

            # Debug: Save best variant's image
            if text and conf >= conf_threshold_rec:
                debug_best_variant_path = self._save_debug_images(img_path, detection_idx, str(variant_name), variants[variant_name])

            if text:
                self.stats['successful_ocr'] += 1
                print(f"[SUCCESS] {Path(img_path).name}: '{text}', '{conf:.4f}', (Length: {len(text)})")
            else:
                self.stats['failed_ocr'] += 1
                print(f"[WARN] No text detected in {Path(img_path).name}")

            return Path(img_path).name, text
        except Exception as e:
            self.stats['failed_ocr'] += 1
            print(f"[ERROR] Detection processing failed for {img_path}: {e}")
            return Path(img_path).name, ""

    def _annotate_image(self, img, box, result):
        x1, y1, x2, y2 = map(int, box[:4])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, result[1], (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    def _save_annotated_images(self, img: np.ndarray, img_path, subdir_name: str="ocr_output_images"):
        output_path = self.output_dir / (subdir_name)
        output_path.mkdir(parents=True, exist_ok=True)

        output_subdir_path = output_path / (Path(img_path).name)
        print(f"[INFO] Annotated image saved to: '{output_subdir_path}'")
        return cv2.imwrite(str(output_subdir_path), img)

    def batch_detect_plates(self, images, image_paths, conf_threshold_det, conf_threshold_rec):
        batch_results = []

        try:
            print(f"[INFO] Running YOLO detection on batch of {len(images)} images...")
            yolo_results = self.yolo_model.predict(
                source=images,
                conf=conf_threshold_det,
                save=False,
                verbose=False
            )
            for img, yolo_result, img_path in zip(images, yolo_results, image_paths):
                self.stats['total_processed'] += 1

                if yolo_result.boxes is None or len(yolo_result.boxes) == 0:
                    self.stats['no_detections'] += 1
                    print(f"[INFO] No plates detected in {Path(img_path).name}")
                    batch_results.append((Path(img_path).name, ""))
                    continue

                boxes = yolo_result.boxes.xyxy.cpu().numpy()
                for i, box in enumerate(boxes):
                    result = self.process_single_detection(img, box, img_path, i, conf_threshold_rec)
                    batch_results.append(result)
                    # Draw bounding box
                    self._annotate_image(img, box, result)

                output_path = self._save_annotated_images(img, img_path)
            return batch_results
        except Exception as e:
            print(f"[ERROR] Batch processing failed: {e}")
            for path in image_paths:
                batch_results.append((Path(path).name, ""))
            return batch_results

    def process_image_batches(self, image_paths: List[str], conf_threshold_det, conf_threshold_rec) -> List[Tuple[str, str]]:
        batch_results = []
        images, valid_paths = [], []

        try:
            for path in image_paths:
                img = cv2.imread(path)
                if img is not None:
                    images.append(img)
                    valid_paths.append(path)
                else:
                    print(f"[WARN] Could not load image: {path}")
                    batch_results.append((Path(path).name, ""))

            if not images:
                return batch_results

            batch_results = self.batch_detect_plates(images, valid_paths, conf_threshold_det, conf_threshold_rec)
            print(f"[DEBUG] Batch results received: {len(batch_results)} results")

            return batch_results
        except Exception as e:
            print(f"[ERROR] Batch processing failed: {e}")
            for path in image_paths:
                batch_results.append((Path(path).name, ""))

    def _seq_batch_processing(self, num_batches, batches, conf_threshold_det, conf_threshold_rec):
        all_results = []

        try:
            for i, batch in enumerate(tqdm(batches, desc="Processing Batches")):
                print(f"[INFO] Processing batch {i + 1}/{num_batches}...")
                results = self.process_image_batches(batch, conf_threshold_det, conf_threshold_rec)
                if results is None:
                    print(f"[ERROR] results returned None —> skipping.")
                    continue
                all_results.extend(results)
                print(f"[STATS] OCR Success: {self.stats['successful_ocr']}, Failures: {self.stats['failed_ocr']}, No Detections: {self.stats['no_detections']}")
        except Exception as e:
            print(f"[ERROR] Batch {i} failed: {e}")

    def _multi_threaded_batch_processing(self, batches, conf_threshold_det, conf_threshold_rec, max_workers):
        all_results = []
        try:
            with ThreadPoolExecutor(max_workers=4) as executor:
                futures = {executor.submit(self.process_image_batches, batch, conf_threshold_det, conf_threshold_rec): i for i, batch in enumerate(batches)}
                for future in as_completed(futures):
                    results = future.result()
                    if results is None:
                        print(f"[ERROR] results returned None —> skipping.")
                        continue
                    all_results.extend(results)
                    print(f"[STATS] OCR Success: {self.stats['successful_ocr']}, Failures: {self.stats['failed_ocr']}, No Detections: {self.stats['no_detections']}")
                return all_results
        except Exception as e:
            print(f"[ERROR] Batch {futures[future]} failed: {e}")

    def process_directory(self, img_dir: str, batch_size: int, conf_threshold_det: int, conf_threshold_rec: int, max_workers: int):
        img_dir = Path(img_dir)
        img_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')

        # Collect image paths
        img_files = [
            str(img_dir / f) for f in os.listdir(img_dir)
            if f.lower().endswith(img_extensions)
        ]
        if not img_files:
            print(f"[WARN] No images found in {img_dir}")
            return

        self.output_dir.mkdir(parents=True, exist_ok=True)

        total_images = len(img_files)
        num_batches = math.ceil(total_images / batch_size)
        all_results = []

        print(f"[INFO] Starting directory processing: {total_images} images in {num_batches} batches")
        batches = [
            img_files[i:i + batch_size]
            for i in range(0, total_images, batch_size)
        ]

        # all_results = self._multi_threaded_batch_processing(batches, conf_threshold_det, conf_threshold_rec, max_workers)
        all_results = self._seq_batch_processing(num_batches, batches, conf_threshold_det, conf_threshold_rec)

        # Save CSV and final stats
        self.save_results(all_results)
        self.print_statistics()
        print(f"[SUCCESS] Dir Processed & Saved")

    def save_results(self, results: List[Tuple[str, str]]):
        try:
            self.csv_path.parent.mkdir(parents=True, exist_ok=True)

            with open(self.csv_path, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(["image_name", "detected_plate"])
                writer.writerows(results)

            print(f"[INFO] Results saved to {self.csv_path}")
        except Exception as e:
            print(f"[ERROR] Failed to save CSV: {e}")

    def print_statistics(self):
        print("\n" + "="*50)
        print("PROCESSING STATISTICS")
        print("="*50)
        print(f"Total Images Processed: {self.stats['total_processed']}")
        print(f"Successful OCR: {self.stats['successful_ocr']}")
        print(f"Failed OCR: {self.stats['failed_ocr']}")
        print(f"No Detections: {self.stats['no_detections']}")

        if self.stats['total_processed']:
            success_rate = (self.stats['successful_ocr'] / self.stats['total_processed']) * 100
            print(f"Success Rate: {success_rate:.2f}%")
        print("="*50)

In [ ]:
if __name__ == "__main__":
    # Configuration
    MODEL_PATH = "/content/models/best.pt"
    IMG_DIR = "/content/data/ocr_processed/images/test/"
    OUTPUT_DIR = "/content/data/ocr_processed/cropped/test/"
    CSV_PATH = "/content/data/detection_results/test_results.csv"
    CONF_THRESH_DET = 0.5
    CONF_THRESH_REC = 0.3
    BATCH_SIZE = 12
    MAX_WORKERS = 4

    # Initialize pipeline
    pipeline = ANPRPipeline(MODEL_PATH, OUTPUT_DIR, CSV_PATH)

    # Process images
    pipeline.process_directory(
        img_dir=IMG_DIR,
        batch_size=BATCH_SIZE,
        conf_threshold_det=CONF_THRESH_DET,
        conf_threshold_rec=CONF_THRESH_REC,
        max_workers=MAX_WORKERS,
    )

In [ ]:
shutil.rmtree('/content/data/ocr_processed/cropped/test/ocr_debug_images')
shutil.rmtree('/content/data/ocr_processed/cropped/train/ocr_debug_images')
shutil.rmtree('/content/data/ocr_processed/cropped/valid/ocr_debug_images')

In [ ]:
shutil.rmtree('/content/results_v1')

In [ ]:
!nvidia-smi

In [ ]:
!pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/

!git clone https://github.com/PaddlePaddle/PaddleOCR
%cd PaddleOCR
!pip install -r requirements.txt

In [ ]:
%pwd

Run Training

In [ ]:
!python tools/train.py -c /content/configs/en_PP-OCRv3_mobile_rec.yml

Evaluate

In [ ]:
!python tools/eval.py -c /content/configs/en_PP-OCRv3_mobile_rec.yml \
-o Global.checkpoints=/content/output/v3_en_mobile/latest

Export Inference Model

In [ ]:
!python tools/export_model.py -c /content/configs/en_PP-OCRv3_mobile_rec.yml \
-o Global.pretrained_model=/content/output/rec_lp/best_accuracy \
Global.save_inference_dir=/content/output/inference/rec_lp/

In [ ]:
# Download logs
!zip -r rec_lp_logs.zip /content/output/rec_lp/

# # Or run visualdl locally on your machine
# visualdl --logdir ./output/rec_lp/

In [ ]:
from google.colab import files

def runs_collab(folder_name="/content/data/ocr_processed/", output_name="ocr_processed_complete.zip"):
    if not os.path.exists(folder_name):
        print(f"[ERROR] Folder '{folder_name}' does not exist.")
        return

    # Create the zip file
    print(f"[INFO] Zipping folder '{folder_name}'...")
    os.system(f"zip -r {output_name} {folder_name}")

    # Download the zip file
    if os.path.exists(output_name):
        print(f"[INFO] Downloading '{output_name}'...")
        files.download(output_name)
    else:
        print("[ERROR] Failed to create zip file.")

runs_collab()